# 🔍 Evaluator-Optimizer in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **The pattern** - Generate, evaluate against explicit criteria, refine, and stop when the grade passes
2. **Criteria the evaluator can apply** - One pass/fail field per criterion, with the verdict computed in code
3. **The loop as a graph** - An evaluator node writes the verdict to state; a pure router function loops back or ends
4. **Bounding the loop** - An iteration cap in the router, proven with an evaluator that never approves
5. **A second model as judge (optional)** - Swap in a Qwen-on-Databricks evaluator without touching the graph

## Where this fits
Module 05's prompt-chaining gate checked a draft once and stopped. Here the check feeds back into the generator, as many times as needed. `02_evaluator_optimizer_lcel.ipynb` rebuilds this loop in LangChain, as a Python loop and as agent middleware.

## Prerequisites
- Conditional edges and state — `modules/05_prompt_chaining/notebooks/01_prompt_chaining_langgraph.ipynb`
- `with_structured_output` — `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- Optional: `DATABRICKS_HOST` and `DATABRICKS_TOKEN` for Part 6 (Qwen via Databricks Foundation Model APIs) — skipped cleanly when absent

---
## 📖 Part 1: The Evaluator-Optimizer Pattern

The evaluator-optimizer workflow represents a sophisticated approach to content generation that mimics human creative processes. This pattern iteratively improves outputs by generating initial content and systematically refining it based on structured feedback from an evaluation model. The workflow creates a feedback loop where each iteration builds upon the previous one, leading to progressively better results.

### Core Concept
The evaluator-optimizer pattern operates on a simple yet powerful principle: generate, evaluate, and refine. This iterative approach ensures that the final output meets specific quality standards before being accepted. The pattern consists of two primary components:
- Generator: Creates initial content or refines existing content based on feedback
- Evaluator: Assesses the quality of generated content against predefined criteria

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Evaluator-optimizer" section |

The pattern pays off when two things hold: there are **clear evaluation criteria**, and feedback **measurably improves** the next attempt. Our demo is deliberately small — a one-line joke — so the loop itself stays visible.

> **Key Insight**: the evaluator is only as useful as its criteria. "Is it funny?" gives the loop nothing to converge on; "does the punchline depend on something specific to the topic?" does.

---
## 📦 Step 1: Environment Setup

Load the API key, then create the two chat models through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: A creative generator and a strict evaluator
# ============================================================================
from aied_helpers import get_openai_llm

# temperature 0.7: the generator must be able to produce a *different* draft on
# each pass - at temperature 0 a refinement can repeat the previous joke.
content_generator = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

# temperature 0: the same draft should get the same grade.
evaluator_llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ Generator: {content_generator.model_name} (temperature={content_generator.temperature})")
print(f"✅ Evaluator: {evaluator_llm.model_name} (temperature={evaluator_llm.temperature})")

---
## 📋 Part 2: Criteria the Evaluator Can Apply

A first version of this notebook asked the evaluator for a single `quality_grade` ("approved" / "needs_improvement") against loose criteria such as "is it genuinely funny?". With `gpt-4o-mini` that loop almost never converged: the evaluator kept finding something that "could be better", even on criteria that were met.

Two changes make the evaluator usable:

### Key Concepts:
- **One field per criterion**: the evaluator answers four yes/no questions instead of one holistic grade. Each answer is easy to check, and a met criterion passes.
- **Checks in code where code can check**: word count is a Python `len(...)`, not an LLM judgement.
- **A computed verdict**: `approved` only when every criterion passes. The model never asserts the grade directly.

In [ ]:
# ============================================================================
# EVALUATION SCHEMA: Per-criterion judgement + a verdict computed in code
# ============================================================================
from pydantic import BaseModel, Field


class JokeReview(BaseModel):
    """One pass/fail judgement per criterion, plus a fix for each failure."""

    clear_punchline: bool = Field(
        description="True if the joke has a setup followed by a distinct punchline that lands the joke"
    )
    topic_specific: bool = Field(
        description=(
            "True if the punchline depends on a concrete feature of agentic AI "
            "(tool calls, planning, memory, loops, autonomy), not a generic computer pun"
        )
    )
    avoids_stock_template: bool = Field(
        description="True if it does NOT use a stock template like 'Why did the ...?', 'X walks into a bar' or 'What do you call ...?'"
    )
    appropriate: bool = Field(description="True if it is appropriate and inoffensive")
    improvement_feedback: str = Field(
        description="For each criterion that is False, a concrete fix. Empty if all pass."
    )


MAX_WORDS = 40  # checked in code: an LLM should not be asked to count


def verdict(review: JokeReview, joke: str) -> tuple[str, str]:
    """Combine the LLM's per-criterion judgement with the code check into one grade."""
    short_enough = len(joke.split()) <= MAX_WORDS
    passed = all(
        [review.clear_punchline, review.topic_specific, review.avoids_stock_template, review.appropriate, short_enough]
    )
    feedback = review.improvement_feedback
    if not short_enough:
        feedback += f" Cut it to at most {MAX_WORDS} words."
    return ("approved" if passed else "needs_improvement"), feedback.strip()


reviewer = evaluator_llm.with_structured_output(JokeReview)

print(f"✅ Criteria: {[f for f in JokeReview.model_fields if f != 'improvement_feedback']} + ≤ {MAX_WORDS} words")

---
## 🔁 Part 3: The Loop as a Graph

### Key Concepts:
- **State** carries the current draft, the latest feedback, the verdict and an iteration counter.
- **Generator node**: the first pass writes from the topic; later passes get the **previous draft plus the feedback**, so they revise instead of starting over.
- **Evaluator node**: calls the reviewer, computes the verdict, and writes it to state.
- **Router**: a pure function of state — no LLM call — that ends on approval or at the iteration cap.

In [ ]:
# ============================================================================
# STATE: What travels around the loop
# ============================================================================
from typing_extensions import TypedDict


class WorkflowState(TypedDict):
    """State management for the evaluator-optimizer workflow"""
    content_topic: str          # The topic or subject for content generation
    generated_content: str      # The current version of generated content
    evaluation_feedback: str    # Feedback from the evaluator
    quality_assessment: str     # "approved" or "needs_improvement"
    iteration: int              # How many drafts have been evaluated


print(f"✅ WorkflowState fields: {list(WorkflowState.__annotations__)}")

In [ ]:
# ============================================================================
# NODES: Generate (or revise) and evaluate
# ============================================================================
def generate_content(state: WorkflowState) -> dict:
    """Generate a first draft, or revise the previous draft using the feedback."""
    if state.get("evaluation_feedback"):
        prompt = (
            f"Here is your previous joke about {state['content_topic']}:\n"
            f"{state['generated_content']}\n\n"
            f"Rewrite it, incorporating this reviewer feedback:\n{state['evaluation_feedback']}\n\n"
            "Make sure to address the specific points mentioned. Reply with the joke only."
        )
    else:
        prompt = f"Create a well-crafted, humorous joke about {state['content_topic']}. Reply with the joke only."

    response = content_generator.invoke(prompt)
    return {"generated_content": response.content}


def make_evaluate_node(reviewer):
    """Build the evaluator node around any runnable that returns a JokeReview."""

    def evaluate_content(state: WorkflowState) -> dict:
        joke = state["generated_content"]
        review = reviewer.invoke("Review this joke about agentic AI systems. Judge each criterion independently; "
    "a criterion that is met passes even if the joke could be better.\n\nJoke: {joke}".format(joke=joke))
        grade, feedback = verdict(review, joke)
        iteration = state.get("iteration", 0) + 1

        print(f"\n🔄 Draft {iteration}: {grade}")
        print(f"   {joke.strip()[:160]}")
        if grade == "needs_improvement":
            print(f"   Feedback: {feedback[:160]}")
        return {"quality_assessment": grade, "evaluation_feedback": feedback, "iteration": iteration}

    return evaluate_content


print("✅ Nodes defined")

### 3.1 🚦 The Router and the Iteration Cap

The router only reads state. Checking the cap here — not inside a node — keeps the stopping rule in one visible place.

In [ ]:
# ============================================================================
# ROUTER: Pure function of state - approve, cap, or loop back
# ============================================================================
MAX_ITERATIONS = 5


def determine_next_action(state: WorkflowState) -> str:
    """Conditional routing function to determine workflow continuation"""
    if state["quality_assessment"] == "approved":
        return "content_approved"
    if state["iteration"] >= MAX_ITERATIONS:
        return "iteration_cap_reached"
    return "content_needs_refinement"


print(f"✅ Router ready (MAX_ITERATIONS={MAX_ITERATIONS})")

### 3.2 🏗️ Building the Graph

The graph is built by a function that takes the reviewer, so Parts 5 and 6 can reuse it with a different evaluator.

In [ ]:
# ============================================================================
# GRAPH: generate -> evaluate -> (END | generate)
# ============================================================================
from langgraph.graph import END, START, StateGraph


def build_workflow(reviewer):
    workflow_builder = StateGraph(WorkflowState)
    workflow_builder.add_node("generate_content", generate_content)
    workflow_builder.add_node("evaluate_content", make_evaluate_node(reviewer))

    workflow_builder.add_edge(START, "generate_content")
    workflow_builder.add_edge("generate_content", "evaluate_content")
    workflow_builder.add_conditional_edges(
        "evaluate_content",
        determine_next_action,
        {
            "content_approved": END,                         # End workflow if content is approved
            "iteration_cap_reached": END,                    # ...or when the budget is spent
            "content_needs_refinement": "generate_content",  # Loop back for refinement
        },
    )
    return workflow_builder.compile()


evaluator_optimizer_workflow = build_workflow(reviewer)
print("✅ Graph compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = evaluator_optimizer_workflow.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

---
## 🧪 Part 4: Running the Loop

Watch for two things in the output: the first draft is rejected with feedback naming the failed criterion, and a later draft passes every criterion and ends the run. (The generator is at temperature 0.7, so the jokes and the number of drafts change from run to run.)

In [ ]:
# ============================================================================
# TEST: Run the evaluator-optimizer loop on one topic
# ============================================================================
def run_evaluator_optimizer(workflow, topic: str) -> dict:
    """Execute the evaluator-optimizer workflow for a given topic"""
    print(f"Starting evaluator-optimizer workflow for topic: {topic}")
    final_state = workflow.invoke({"content_topic": topic, "iteration": 0})

    print("\n" + "=" * 50)
    print("EVALUATOR-OPTIMIZER WORKFLOW RESULTS")
    print("=" * 50)
    print(f"Topic: {final_state['content_topic']}")
    print(f"Drafts evaluated: {final_state['iteration']}")
    print(f"Quality Status: {final_state['quality_assessment']}")
    print(f"Final Content: {final_state['generated_content']}")
    return final_state


result = run_evaluator_optimizer(evaluator_optimizer_workflow, "Agentic AI systems")

---
## 🛑 Part 5: Bounding the Loop

A structured verdict does not guarantee the evaluator will *ever* approve. To prove the cap works, we build the same graph with a stub reviewer that rejects every draft (no LLM call for the review; the generator still runs). Without the `iteration_cap_reached` branch this cell would never finish.

In [ ]:
# ============================================================================
# STUBBORN EVALUATOR: Never approves - the router's cap must stop the loop
# ============================================================================
from langchain_core.runnables import RunnableLambda

never_satisfied = RunnableLambda(
    lambda _prompt: JokeReview(
        clear_punchline=False,
        topic_specific=False,
        avoids_stock_template=False,
        appropriate=True,
        improvement_feedback="Try a completely different angle.",
    )
)

capped = run_evaluator_optimizer(build_workflow(never_satisfied), "Agentic AI systems")
print(f"\n✅ Loop stopped by the cap: {capped['iteration'] == MAX_ITERATIONS and capped['quality_assessment'] == 'needs_improvement'}")

---
## ☁️ Part 6 (Optional): A Different Model as the Judge

Using the same model to write and to judge can make it lenient toward its own style. A common variant is an evaluator from a different model family. Because the graph takes the reviewer as a parameter, swapping in **Qwen 3.5 (`system.ai.qwen35-122b-a10b`) on Databricks** only changes how the reviewer is built.

`get_databricks_llm` talks to the Databricks AI Gateway through its OpenAI-compatible API. `with_structured_output` does not work there for this model: the default JSON-schema mode fails because Qwen returns a reasoning block next to the answer, and the function-calling mode sends `parallel_tool_calls`, which the gateway rejects. So the cell builds the same thing by hand: bind `JokeReview` as a tool, **force** the call with `tool_choice`, and parse the arguments with `PydanticToolsParser`. The reviewer still returns a `JokeReview`, so the graph is unchanged.

This cell runs only when `DATABRICKS_HOST` and `DATABRICKS_TOKEN` are set; otherwise it prints a skip line.

In [ ]:
# ============================================================================
# DATABRICKS EVALUATOR (OPTIONAL): Same graph, Qwen as the judge
# ============================================================================
import os

from aied_helpers import get_databricks_llm
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

DATABRICKS_JUDGE_MODEL = "system.ai.qwen35-122b-a10b"

if not (os.getenv("DATABRICKS_HOST") and os.getenv("DATABRICKS_TOKEN")):
    print("⏭️  Skipped: DATABRICKS_HOST / DATABRICKS_TOKEN not set - the Databricks evaluator demo needs them")
else:
    try:
        qwen_llm = get_databricks_llm(model=DATABRICKS_JUDGE_MODEL)
        # Forced tool call + parser = structured output that works on the Databricks gateway
        qwen_reviewer = qwen_llm.bind_tools([JokeReview], tool_choice="JokeReview") | PydanticToolsParser(
            tools=[JokeReview], first_tool_only=True
        )
        run_evaluator_optimizer(build_workflow(qwen_reviewer), "Agentic AI systems")
    except Exception as exc:  # optional demo: report, don't crash the notebook
        print(f"⚠️ Databricks evaluator failed: {type(exc).__name__}: {str(exc)[:200]}")

---
## 📝 Summary

In this notebook, we learned:

### 1. The pattern
- Generate, evaluate, refine: the evaluator's feedback becomes part of the next generation prompt, together with the previous draft.
- It fits when criteria are clear and feedback measurably improves the next attempt.

### 2. Criteria the evaluator can apply
- One boolean per criterion beats one holistic grade: each question is easy to answer and a met criterion passes.
- Check in code what code can check (length); compute the verdict from the parts.

### 3. The loop as a graph
- The evaluator node writes the verdict to state; a pure router function reads it and loops back to `generate_content` or ends.

### 4. Bounding the loop
- `iteration` in state plus a cap in the router guarantees termination — proven with an evaluator that never approves.

### 5. A second model as judge
- Passing the reviewer into `build_workflow` makes the judge swappable (Qwen on Databricks here) without touching the graph.

### Next Steps
- **`02_evaluator_optimizer_lcel.ipynb`**: the same loop in LangChain — a Python driver around LCEL chains, and `after_model` middleware with `jump_to`